# WBCBench 2026 — Aşama 11B
RETFound Full FT, **Seeds 1 & 7**, aynı kilitli protokol: max12/patience3. Resume destekli; test kullanılmaz.


In [1]:

# Aşama 11B — RETFound Seeds 1 & 7
from pathlib import Path
import random,time,json,csv,importlib.util,numpy as np,torch
from torch import nn
from torch.utils.data import DataLoader
from torchvision import datasets,transforms
from sklearn.metrics import f1_score,balanced_accuracy_score,accuracy_score
ROOT=Path.cwd(); TR=ROOT/"wbc_project/data/processed/phase2_train"; VA=ROOT/"wbc_project/data/processed/phase2_eval"
MF=ROOT/"RETFound/models_vit.py"; CK=ROOT/"wbc_project/checkpoints/RETFound_cfp_weights.pth"
assert TR.exists() and VA.exists() and MF.exists() and CK.exists()
device=torch.device("cuda"); assert torch.cuda.is_available()
MAXE,PAT,ACCUM=12,3,4
tt=transforms.Compose([transforms.Resize((224,224)),transforms.RandomHorizontalFlip(.5),transforms.RandomRotation(10),transforms.ToTensor(),transforms.Normalize([.485,.456,.406],[.229,.224,.225])])
vt=transforms.Compose([transforms.Resize((224,224)),transforms.ToTensor(),transforms.Normalize([.485,.456,.406],[.229,.224,.225])])
def model_new():
 s=importlib.util.spec_from_file_location("rv",MF); m=importlib.util.module_from_spec(s); s.loader.exec_module(m)
 return m.RETFound_mae(num_classes=13,drop_path_rate=0.,global_pool=True).to(device)
@torch.inference_mode()
def val(m,L,c):
 m.eval();Y=[];P=[];tot=n=0
 for x,y in L:
  x,y=x.to(device),y.to(device)
  with torch.autocast("cuda",dtype=torch.float16): z=m(x); q=c(z,y)
  tot+=q.item()*len(y);n+=len(y);Y+=y.cpu().tolist();P+=z.argmax(1).cpu().tolist()
 return tot/n,f1_score(Y,P,average="macro",zero_division=0),balanced_accuracy_score(Y,P),accuracy_score(Y,P)
for SEED in [1,7]:
 random.seed(SEED);np.random.seed(SEED);torch.manual_seed(SEED);torch.cuda.manual_seed_all(SEED)
 torch.backends.cudnn.deterministic=True;torch.backends.cudnn.benchmark=False
 td=datasets.ImageFolder(TR,tt);vd=datasets.ImageFolder(VA,vt);g=torch.Generator().manual_seed(SEED)
 TL=DataLoader(td,1,shuffle=True,num_workers=0,pin_memory=True,generator=g);VL=DataLoader(vd,2,shuffle=False,num_workers=0,pin_memory=True)
 m=model_new();c=nn.CrossEntropyLoss();o=torch.optim.AdamW(m.parameters(),lr=5e-6,weight_decay=1e-4);sc=torch.amp.GradScaler("cuda")
 O=ROOT/f"outputs/final/retfound_fullft/seed{SEED}";O.mkdir(parents=True,exist_ok=True)
 LAST=O/f"retfound_fullft_seed{SEED}_last_resume.pt";BEST=O/f"retfound_fullft_seed{SEED}_best.pt"
 hist=[];best=-1.;bad=0;start=1
 if LAST.exists():
  r=torch.load(LAST,map_location="cpu",weights_only=False);m.load_state_dict(r["model_state_dict"]);o.load_state_dict(r["optimizer_state_dict"]);sc.load_state_dict(r["scaler_state_dict"])
  hist=r["history"];best=float(r["best_val_macro_f1"]);start=int(r["epoch"])+1
  fs=[x["macro_f1"] for x in hist];bad=len(hist)-1-int(np.argmax(fs));print(f"SEED{SEED} RESUME {start-1}->{start}, best={best:.4f}")
 else:
  r=torch.load(CK,map_location="cpu",weights_only=False);sd=r.get("model",r.get("state_dict",r));ms=m.state_dict()
  good={k:v for k,v in sd.items() if k in ms and hasattr(v,"shape") and v.shape==ms[k].shape and not k.startswith("head.")}
  m.load_state_dict(good,strict=False);print(f"SEED{SEED}: CFP exact compatible keys={len(good)}")
 cfg={"seed":SEED,"max_epochs":12,"patience":3,"lr":5e-6,"weight_decay":1e-4,"batch":1,"grad_accum":4,"test_used":False}
 for ep in range(start,MAXE+1):
  if bad>=PAT:break
  print(f"\nSEED {SEED} EPOCH {ep}/12");m.train();o.zero_grad(set_to_none=True);tot=n=0;t=time.time()
  for i,(x,y) in enumerate(TL,1):
   x,y=x.to(device),y.to(device)
   with torch.autocast("cuda",dtype=torch.float16):z=m(x);raw=c(z,y);loss=raw/ACCUM
   sc.scale(loss).backward()
   if i%ACCUM==0 or i==len(TL):sc.step(o);sc.update();o.zero_grad(set_to_none=True)
   tot+=raw.item()*len(y);n+=len(y)
   if i%2000==0:print(f" batch {i}/{len(TL)} loss={tot/n:.4f}")
  vl,f1,ba,acc=val(m,VL,c);row={"epoch":ep,"train_loss":tot/n,"val_loss":vl,"macro_f1":f1,"balanced_acc":ba,"accuracy":acc,"minutes":(time.time()-t)/60};hist.append(row)
  print(f"Macro-F1={f1:.4f} | BA={ba:.4f} | Acc={acc:.4f} | {row['minutes']:.1f} dk")
  if f1>best:
   best=f1;bad=0;torch.save({"model_state_dict":m.state_dict(),"classes":td.classes,"seed":SEED,"epoch":ep,"val_macro_f1":best,"config":cfg},BEST);print("✓ Yeni BEST")
  else:bad+=1;print(f"İyileşme yok {bad}/3")
  torch.save({"model_state_dict":m.state_dict(),"optimizer_state_dict":o.state_dict(),"scaler_state_dict":sc.state_dict(),"epoch":ep,"best_val_macro_f1":best,"bad_epochs":bad,"history":hist,"config":cfg},LAST)
  (O/"history.json").write_text(json.dumps(hist,indent=2),encoding="utf-8")
  with open(O/"history.csv","w",newline="",encoding="utf-8") as f:w=csv.DictWriter(f,fieldnames=hist[0].keys());w.writeheader();w.writerows(hist)
 print(f"\nSEED {SEED} SONUÇ: best val Macro-F1={best:.4f}, epochs={len(hist)}")
 del m,o,sc,TL,VL;torch.cuda.empty_cache()
print("\n✓ SEEDS 1 & 7 BİTTİ. TEST SPLIT KULLANILMADI.")


C:\Users\sarah\.conda\envs\retfound\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


SEED1 RESUME 7->8, best=0.6215

SEED 1 SONUÇ: best val Macro-F1=0.6215, epochs=7
SEED7 RESUME 4->5, best=0.5778

SEED 7 EPOCH 5/12
 batch 2000/49794 loss=0.1640
 batch 4000/49794 loss=0.1700
 batch 6000/49794 loss=0.1684
 batch 8000/49794 loss=0.1740
 batch 10000/49794 loss=0.1736
 batch 12000/49794 loss=0.1688
 batch 14000/49794 loss=0.1648
 batch 16000/49794 loss=0.1637
 batch 18000/49794 loss=0.1642
 batch 20000/49794 loss=0.1643
 batch 22000/49794 loss=0.1610
 batch 24000/49794 loss=0.1574
 batch 26000/49794 loss=0.1559
 batch 28000/49794 loss=0.1561
 batch 30000/49794 loss=0.1541
 batch 32000/49794 loss=0.1532
 batch 34000/49794 loss=0.1523
 batch 36000/49794 loss=0.1530
 batch 38000/49794 loss=0.1525
 batch 40000/49794 loss=0.1516
 batch 42000/49794 loss=0.1515
 batch 44000/49794 loss=0.1514
 batch 46000/49794 loss=0.1496
 batch 48000/49794 loss=0.1492
Macro-F1=0.5589 | BA=0.5568 | Acc=0.9182 | 129.4 dk
İyileşme yok 1/3

SEED 7 EPOCH 6/12
 batch 2000/49794 loss=0.1099
 batch 4000